In [1]:
import logging

import joblib
import numpy as np
import pandas as pd

from beir import LoggingHandler
from beir.retrieval import models
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch

from gensim.models.doc2vec import Doc2Vec
from gensim.models.fasttext import FastText
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from stemmer import Stemmer, tokenize

/home/fahmi/freelance/project-2023-amsearch/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

In [3]:
## EVAL PARAMS
MODEL_NAME = "../models/ams-fasttext.gensim" # "multi-qa-mpnet-base-dot-v1"
STEMMING_AT_EVAL = False

VOCAB_PATH = "../data/sundabaru1-vocab.txt"
BEIR_DATA_PATH = "../data/cleaned"
BEIR_QRELS_PATH = "../data/cleaned/qrels.tsv"

## Data Loading

In [4]:
stemmer = Stemmer(VOCAB_PATH)

In [5]:
def stem_sentence(text: str) -> str:
    return " ".join([stemmer.stem_ams(t) for t in tokenize(text)])

def stem_corpus(item: dict[str, str]):
    return {k: stem_sentence(v) for k, v in item.items()}

In [6]:
corpus, queries, qrels = GenericDataLoader(data_folder=BEIR_DATA_PATH, qrels_file=BEIR_QRELS_PATH).load_custom()
if STEMMING_AT_EVAL:
    queries = stem_corpus(queries)
    corpus = {k: stem_corpus(v) for k, v in corpus.items()}

2025-05-09 12:24:45 - Loading Corpus...


100%|██████████| 1499/1499 [00:00<00:00, 151114.30it/s]

2025-05-09 12:24:45 - Loaded 1499 Documents.
2025-05-09 12:24:45 - Doc Example: {'text': 'bismillah yuga lampah balukar janglar meunang kahayang balukar sasar pedar ringkang sing jembar sabar tong jadi hambar ihtér raksa rasa ukir pikir mun gering tong rungsing mun cageur tong badeur ulah ceurik jadi jalma leutik ulah sombong abong di gedong lain batur kabéh gelé dulur néang halal awur amal ulah lieur ku madu dunya jung sanding ka nu agung gusti wéti ngarti nu sajati pariksa ati', 'title': 'JEMBAR SABAR'}
2025-05-09 12:24:45 - Loading Queries...
2025-05-09 12:24:45 - Loaded 7491 Queries.
2025-05-09 12:24:45 - Query Example: apa maksud dari bismillah yuga lampah


## Evaluate Model

https://github.com/beir-cellar/beir/wiki/Evaluate-your-custom-model

In [7]:
class AMSTokenizer:
    def __init__(self, stm: Stemmer):
        self.stemmer = stm

    def __call__(self, doc):
        return [self.stemmer.stem_ams(word) for word in tokenize(doc)]

### Create Evaluator for BoW, TF-IDF, Doc2Vec, FastText

In [14]:
class AMSCustomModelEvaluator:
    def __init__(self, model_name: str, stemmer: Stemmer=None, **kwargs):
        self.model_name = model_name
        self.stemmer = stemmer

        if "bow" in model_name or "tfidf" in model_name:
            self.model = joblib.load(self.model_name)
            if "stem" in self.model_name:
                self.model.tokenizer = AMSTokenizer(self.stemmer)
        elif "doc2vec" in model_name:
            self.model = Doc2Vec.load(model_name)
        elif "fasttext" in model_name:
            self.model = FastText.load(model_name)
        else:
            raise ValueError("Invalid model name")

    def encode_queries(self, queries: list[str], batch_size: int, **kwargs) -> np.ndarray:
        if isinstance(self.model, CountVectorizer) or isinstance(self.model, TfidfVectorizer):
            return self.model.transform(queries).todense().astype(float)
        
        if isinstance(self.model, Doc2Vec):
            return np.array([self.model.infer_vector(tokenize(sentence)) for sentence in queries]).astype(float)
            
        if isinstance(self.model, FastText):
            return np.array([self.model.wv.get_sentence_vector(sentence) for sentence in queries]).astype(float)

    def encode_corpus(self, corpus: list[dict[str, str]], batch_size: int, **kwargs) -> np.ndarray:
        extracted_corpus = [row["title"] + " " + row["text"] for row in corpus]
        return self.encode_queries(extracted_corpus, batch_size, **kwargs)

In [ ]:
model = DenseRetrievalExactSearch(AMSCustomModelEvaluator(MODEL_NAME, stemmer), batch_size=16)
model

2025-05-09 12:26:31 - loading FastText object from ../models/ams-fasttext.gensim
2025-05-09 12:26:31 - loading wv recursively from ../models/ams-fasttext.gensim.wv.* with mmap=None
2025-05-09 12:26:31 - loading vectors_ngrams from ../models/ams-fasttext.gensim.wv.vectors_ngrams.npy with mmap=None
2025-05-09 12:26:31 - setting ignored attribute buckets_word to None
2025-05-09 12:26:31 - setting ignored attribute vectors to None
2025-05-09 12:26:31 - setting ignored attribute cum_table to None
2025-05-09 12:26:31 - FastText lifecycle event {'fname': '../models/ams-fasttext.gensim', 'datetime': '2025-05-09T12:26:31.942044', 'gensim': '4.3.2', 'python': '3.11.12 (main, Apr  9 2025, 04:04:00) [Clang 20.1.0 ]', 'platform': 'Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'event': 'loaded'}


### Create Evaluator for Sentence Transformers

In [ ]:
model = DenseRetrievalExactSearch(models.SentenceBERT(MODEL_NAME), batch_size=16)
model

## Evaluate

In [16]:
retriever = EvaluateRetrieval(model, score_function="cos_sim")  # dot or cos_sim
results = retriever.retrieve(corpus, queries)

2025-05-09 12:26:33 - Encoding Queries...
2025-05-09 12:26:38 - Sorting Corpus by document length (Longest first)...
2025-05-09 12:26:38 - Encoding Corpus in batches... Warning: This might take a while!
2025-05-09 12:26:38 - Scoring Function: Cosine Similarity (cos_sim)
2025-05-09 12:26:38 - Encoding Batch 1/1...


In [11]:
#### Evaluate your model with NDCG@k, MAP@K, Recall@K and Precision@K  where k = [1,3,5,10,100,1000]
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, retriever.k_values)
metrics = [
    {
        "model": MODEL_NAME, 
        "stemming": STEMMING_AT_EVAL,
        "metric": k.split("@")[0], 
        "k": k.split("@")[1], 
        "value": v
    } for col in [ndcg, _map, recall, precision] for k, v in col.items()
]

2025-05-09 12:25:18 - For evaluation, we ignore identical query and document ids (default), please explicitly set ``ignore_identical_ids=False`` to ignore this.
2025-05-09 12:25:20 - 

2025-05-09 12:25:20 - NDCG@1: 0.0143
2025-05-09 12:25:20 - NDCG@3: 0.0237
2025-05-09 12:25:20 - NDCG@5: 0.0293
2025-05-09 12:25:20 - NDCG@10: 0.0373
2025-05-09 12:25:20 - NDCG@100: 0.0746
2025-05-09 12:25:20 - NDCG@1000: 0.1441
2025-05-09 12:25:20 - 

2025-05-09 12:25:20 - MAP@1: 0.0143
2025-05-09 12:25:20 - MAP@3: 0.0212
2025-05-09 12:25:20 - MAP@5: 0.0243
2025-05-09 12:25:20 - MAP@10: 0.0276
2025-05-09 12:25:20 - MAP@100: 0.0335
2025-05-09 12:25:20 - MAP@1000: 0.0354
2025-05-09 12:25:20 - 

2025-05-09 12:25:20 - Recall@1: 0.0143
2025-05-09 12:25:20 - Recall@3: 0.0308
2025-05-09 12:25:20 - Recall@5: 0.0446
2025-05-09 12:25:20 - Recall@10: 0.0692
2025-05-09 12:25:20 - Recall@100: 0.2646
2025-05-09 12:25:20 - Recall@1000: 0.8545
2025-05-09 12:25:20 - 

2025-05-09 12:25:20 - P@1: 0.0143
2025-05-09 12:25:20

In [13]:
df_metrics = pd.DataFrame(metrics)
df_metrics.to_csv(f"eval.csv", index=None)

df_metrics

,model,stemming,metric,k,value
0,../models/ams-fasttext.gensim,False,NDCG,1,0.01428
1,../models/ams-fasttext.gensim,False,NDCG,3,0.02368
2,../models/ams-fasttext.gensim,False,NDCG,5,0.02931
3,../models/ams-fasttext.gensim,False,NDCG,10,0.03726
4,../models/ams-fasttext.gensim,False,NDCG,100,0.07455
5,../models/ams-fasttext.gensim,False,NDCG,1000,0.14414
6,../models/ams-fasttext.gensim,False,MAP,1,0.01428
7,../models/ams-fasttext.gensim,False,MAP,3,0.02123
8,../models/ams-fasttext.gensim,False,MAP,5,0.02434
9,../models/ams-fasttext.gensim,False,MAP,10,0.02762
